<div class = "alert alert-info"> <h2 align = "center"> <b> Cross-asset Macro Overlay </b> </h2> 
<h3 align = center> <font  color = "black"> Modélisation Dynamique <font/> </h3>

</div>

### __Contexte__

Les portefeuilles multi-actifs sont traditionnellement construits sur des fondations quantitatives (corrélations, volatilités, betas). Cependant, une grande partie des mouvements de marché est macro-driven (inflation, taux, croissance, politique monétaire).

Deux sources d’information sont cruciales mais souvent exploitées séparément :

- Marché des options de taux → anticipe les mouvements futurs des banques centrales et l’incertitude (via vol implicite, skew, term structure).

- Flux de nouvelles macroéconomiques → messages textuels des banques centrales, données économiques, événements géopolitiques.

__Le projet vise à fusionner ces deux sources pour générer un overlay dynamique cross-asset, capable de repositionner le portefeuille face aux chocs macro.__



### __Objectifs principaux :__

- Construire un overlay dynamique cross-asset basé sur : Signaux textuels macro (news, communiqués, discours de banques centrales); Options rates (volatilité implicite, skew, convexité).

- Ajuster l’allocation en fonction de régimes de marché identifiés via Markov Switching / State-space.

- Quantifier l’impact des news sur la performance du portefeuille (news-to-portfolio attribution).



### __Modélisation dynamique__

#### __Import des librairies et des données__

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pypfopt import EfficientFrontier, risk_models, expected_returns
import cvxpy as cp
import matplotlib.pyplot as plt
import random
import os


# Standard reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


# Load synced data with factors from previous step (assumes exists; adjust path)
df = pd.read_parquet('./data/synced_with_factors.parquet')

# Select relevant columns
# Latent states: macro_signal = latent_macro_factor, rates_signal = latent_options_factor
state_vars = ['latent_macro_factor', 'latent_options_factor'] # 'pca_factor_1', 'pca_factor_2']

# Observations: Cross-asset returns (select key assets; adjust as needed)
obs_vars = ['^GSPC', '^STOXX50E', '^IRX', '^TNX', 'EURUSD=X', 'JPYUSD=X', 'CL=F', 'GC=F']  # From returns.csv

WINDOW = 252
MIN_ROWS = 60

# Drop NaNs and align
df_model = df[state_vars + obs_vars].dropna()
print(df_model.head())

#### __State-space / Kalman Filter__

Un modèle state-space permet de distinguer entre :

- Des états latents (non observables directement mais qui reflètent la "vraie" dynamique économique/financière, ex : facteur macro global, facteur lié aux options rates)

- Des observations bruyantes (les rendements multi-actifs quotidiens, vol implicite observée, news sentiment, etc.)

Formellement: $X_t = A X_{t-1} + w_t (transition, \: état \: latent)$ ; $Y_t = C X_t + v_t \:\:(mesure, \:\:observation\: avec\: bruit)$

Avec: 
- $X_t$  : Facteurs latents (macro_signal et rates_signal)
- $Y_t$  : Rendements observés sur cross asset (S&P500, EuroStoxx50, Bonds, FX, Commodities)
- $w_t \:, \: v_t$  : Bruits / Incertitudes

Ce filtre estime les états latents en temps réel : il combine les signaux macro textuels et les prix/options observés pour inférer un facteur "macro latent" et un facteur "rates latent" plus propres que les données brutes; Cela permet de « filtrer » le bruit de marché. D'autre part, il permet de faire de la prévision: à chaque date $t$, il donne une estimation de $X_t$ et une prévision de $X_{t+1}$; On obtient donc une vision dynamique, utile pour anticiper les régimes.

In [ ]:
# 4a) State-Space / Kalman Filter - Custom Model
class CustomStateSpace(sm.tsa.statespace.MLEModel):
    def __init__(self, endog, k_states=2):
        # Initialize superclass
        super(CustomStateSpace, self).__init__(
            endog=endog, k_states=k_states, initialization='diffuse'
        )

        # Initial matrices (will be updated in update)
        self['selection'] = np.eye(k_states)  # Fixed

    @property
    def param_names(self):
        # Params: A (4), Q diag (2), C (n_obs * k_states), R diag (n_obs)
        n_obs = self.k_endog
        return (
            [f'a{i}{j}' for i in range(1,3) for j in range(1,3)] + 
            [f'q{i}' for i in range(1,3)] + 
            [f'c{o}{s}' for o in range(1, n_obs+1) for s in range(1,3)] + 
            [f'r{o}' for o in range(1, n_obs+1)]
        )

    @property
    def start_params(self):
        # Initial guesses: A identity*0.9, Q 0.1, C random small, R 0.05
        n_obs = self.k_endog
        a_params = np.eye(2).flatten() * 0.9
        q_params = np.ones(2) * 0.1
        c_params = np.random.normal(0, 0.1, n_obs * 2)
        r_params = np.ones(n_obs) * 0.05
        return np.concatenate([a_params, q_params, c_params, r_params])

    def transform_params(self, unconstrained):
        # Make variances positive (Q and R)
        constrained = unconstrained.copy()
        n_obs = self.k_endog
        # Q: indices 4:6
        constrained[4:6] = constrained[4:6]**2
        # R: last n_obs
        constrained[-n_obs:] = constrained[-n_obs:]**2
        return constrained

    def untransform_params(self, constrained):
        unconstrained = constrained.copy()
        n_obs = self.k_endog
        unconstrained[4:6] = constrained[4:6]**0.5
        unconstrained[-n_obs:] = constrained[-n_obs:]**0.5
        return unconstrained

    def update(self, params, **kwargs):
        params = super(CustomStateSpace, self).update(params, **kwargs)
        k_states = self.k_states
        n_obs = self.k_endog

        # A: first 4 params -> 2x2
        self['transition', :, :] = params[:4].reshape(2, 2)

        # Q: next 2, diag
        self['state_cov'] = np.diag(params[4:6])

        # C: next 14 (7*2), 7x2
        self['design', :, :] = params[6:6 + n_obs*2].reshape(n_obs, 2)

        # R: last 7, diag
        self['obs_cov'] = np.diag(params[-n_obs:])

# Create and fit
n_obs = len(obs_vars)
model_kf = CustomStateSpace(endog=df_model[obs_vars])
results_kf = model_kf.fit(maxiter=50, disp=False)  # disp=False to reduce output

# Extract filtered states
df_filtered_states = pd.DataFrame(results_kf.filtered_state.T, index=df_model.index, columns=state_vars)
df['filtered_macro_signal'] = df_filtered_states['latent_macro_factor']
df['filtered_rates_signal'] = df_filtered_states['latent_options_factor']


# Plot filtered states
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['filtered_macro_signal'], label='Filtered Macro Signal')
plt.plot(df.index, df['filtered_rates_signal'], label='Filtered Rates Signal')
plt.title('Filtered Latent States from Kalman Filter')
plt.legend()
plt.show()

#### __Markov Switching__


Le modèle MarkovRegression à 4 régimes `(k_regimes=4)` cherche à capturer les changements structurels dans la dynamique des rendements, en utilisant comme variables explicatives les signaux filtrés par Kalman (macro + options).

Le modèle suppose qu’à chaque instant $t$, le marché est dans un régime latent $S_t \in \{0, 1, 2, 3\}$.

Chaque régime est associé à une dynamique propre (moyenne des rendements, volatilité, sensibilité aux facteurs macro). La probabilité de transition d’un régime à un autre est donnée par une matrice de transition P.

L’objectif est donc d’identifier des régimes cohérents économiquement et statistiquement.


| **Régime**                                    | **Caractéristiques statistiques**                                                                                                      | **Signaux macro (NLP + options)**                                                                                       | **Allocation type (overlay)**                                                                                                                         |
| --------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Dovish (accommodant)**                  | - Actions ↑ (SP500, STOXX)  <br> - Taux ↓ (obligations ↑) <br> - Volatilité faible <br> - Corrélation actions–obligations négative | - NLP : discours « dovish », mots-clés accommodants <br> - Options rates : volatilité implicite faible, skew neutre     | **Pro-risque** : <br> surpondérer actions & crédit <br> duration longue en obligations <br> commodities neutres                                       |
| **Hawkish (resserrement)**                | - Actions ↓ <br> - Taux ↑ (Treasuries yield ↑) <br> - Volatilité modérée <br> - USD parfois fort                                       | - NLP : discours « hawkish », ton restrictif <br> - Options rates : skew haussier sur taux, convexité ↑                 | **Défensif** : <br> réduire actions <br> duration courte en fixed income <br> FX : USD overweight <br> commodities sous-pondérées                     |
| **Inflation scare (choc inflationniste)** | - Actions ↓ <br> - Obligations ↓ (corrélation actions-obligations positive) <br> - Commodities ↑ (oil, gold) <br> - Volatilité ↑   | - NLP : hausse fréquence « inflation », « price pressures » <br> - Options : convexité accrue, skew sur inflation swaps | **Inflation hedge** : <br> overweight commodities (oil, gold) <br> sous-pondérer actions + bonds <br> overweight devises liées aux matières premières |
| **Growth scare (crainte récession)**      | - Actions ↓↓↓ (drawdown) <br> - Obligations ↑ (flight-to-quality) <br> - Or ↑ <br> - JPY ↑ <br> - Volatilité très élevée               | - NLP : langage « downturn », « unemployment », « slowdown » <br> - Options : skew baissier sur actions, vol bonds ↑    | **Récession hedge** : <br> overweight Treasuries, gold, JPY <br> réduire actions & commodities cycliques                                              |


In [ ]:
# 4b) Markov Switching

# Identify regimes: Use MarkovRegression on a key observation, e.g., ^GSPC returns
# Exog: filtered states (macro and rates signals)
# k_regimes=4 for hawkish/dovish, inflation scare, growth scare

endog = df['^GSPC']  # Example: SP500 returns as dependent
exog = df[['filtered_macro_signal', 'filtered_rates_signal']] #, 'pca_factor_1', 'pca_factor_2']]

# Markov Regression with switching mean and variance
model_ms = sm.tsa.MarkovRegression(endog=endog, k_regimes=4, trend='c', exog=exog, switching_variance=True)
results_ms = model_ms.fit()
print(results_ms.summary())
# Extract regime probabilities
regime_probs = results_ms.smoothed_marginal_probabilities
df_regimes = pd.DataFrame(regime_probs, index=endog.index)
df_regimes.columns = ['Regime0_Prob', 'Regime1_Prob', 'Regime2_Prob', 'Regime3_Prob']

# Assign most likely regime
df['regime'] = df_regimes.idxmax(axis=1).str.extract('(\d+)').astype(int)  # 0: dovish, 1: hawkish, 2: inflation scare, 3: growth scare (interpret post-fit)

# Regime summary: Mean returns/vol per regime
regime_summary = df_model[obs_vars].groupby(df['regime']).agg(['mean', 'std'])
print("Regime Summary (Mean and Std of Returns):")
print(regime_summary)

# Plot regime probabilities
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
for i in range(4):
    axes[i].plot(df_regimes.index, df_regimes[f'Regime{i}_Prob'], label=f'Regime {i}')
    axes[i].set_title(f'Probability of Regime {i}')
plt.tight_layout()
plt.show()

#### __Overlay dynamique__


En asset management, un overlay est une stratégie qui s’ajoute à une allocation de base (« core portfolio »), afin d'ajuster dynamiquement l’exposition, de réduire les risques (hedging), et d'exploiter des opportunités tactiques.

L’overlay vient ajuster les poids de l’allocation de base en fonction des conditions récentes (volatilité, corrélations, rendements attendus).
Comme résultat, nous obtenons un portefeuille qui évolue dans le temps en fonction du régime macro, mais reste optimisé et protégé.





In [ ]:
# 4c) Overlay dynamique
# Mapping regime → base weights (adjust based on interpretation)
# Then optimize: max Sharpe with constraints weights [0,1], sum=1
# Use PyPortfolioOpt for simplicity

def get_base_weights(regime):
    if regime == 0:  # Dovish: Favor growth assets
        return np.array([0.3, 0.2, 0.1, 0.1, 0.1, 0.1, 0.05, 0.05])  # SP500, STOXX, IRX, TNX, EURUSD, JPYUSD, CL, GC
    elif regime == 1:  # Hawkish: Defensive     
        return np.array([0.1, 0.1, 0.2, 0.3, 0.05, 0.05, 0.1, 0.1])
    elif regime == 2:  # Inflation scare: Commodities heavy
        return np.array([0.1, 0.1, 0.1, 0.1, 0.05, 0.05, 0.2, 0.3])
    elif regime == 3:  # Growth scare: Safe havens
        return np.array([0.1, 0.1, 0.2, 0.3, 0.1, 0.1, 0.05, 0.05])
    else:              
        return np.array([1/len(obs_vars)] * len(obs_vars))  # Equal

# Apply base weights per day
df['base_weights'] = df['regime'].apply(get_base_weights)

# Rolling optimization: For each day, optimize from historical data
# Prototype: Use last 252 days (1 year) for mu/cov



In [ ]:
#------------------------------------------------------------------------------------------------------------------------

def winsorize_series(s, lower_q=0.005, upper_q=0.995):
    if s.dropna().empty:
        return s
    lo = s.quantile(lower_q)
    hi = s.quantile(upper_q)
    return s.clip(lower=lo, upper=hi)

def prepare_returns_window(window_df, max_abs=1e6, min_rows=30, col_nan_thresh=0.5):
    w = window_df.copy().astype(float)
    w.replace([np.inf, -np.inf], np.nan, inplace=True)
    thresh = int((1 - col_nan_thresh) * len(w))
    if thresh <= 0:
        thresh = 1
    w = w.dropna(axis=1, thresh=thresh)
    if w.shape[1] == 0:
        return None
    for col in w.columns:
        w[col] = winsorize_series(w[col], 0.005, 0.995)
    w = w.clip(lower=-max_abs, upper=max_abs)
    w = w.fillna(method='ffill').fillna(method='bfill')
    if w.isna().any().any():
        w = w.fillna(0.0)
    if w.shape[0] < max(min_rows, w.shape[1] + 2):
        return None
    return w


def safe_cov_and_mu(window_df, target_cols, freq=252):
    w_clean = prepare_returns_window(window_df[target_cols], min_rows=MIN_ROWS)
    if w_clean is None:
        return None, None, []
    try:
        mu = expected_returns.mean_historical_return(w_clean, frequency=freq)
    except Exception:
        mu = w_clean.mean() * freq
    valid_cols = w_clean.columns.tolist()
    # shrinkage
    try:
        cs = risk_models.CovarianceShrinkage(w_clean)
        S = cs.ledoit_wolf()
        S = (S + S.T) / 2.0
        S += np.eye(S.shape[0]) * 1e-8
    except Exception:
        S = w_clean.cov().values
        S = (S + S.T) / 2.0
        vals, vecs = np.linalg.eigh(S)
        eps = 1e-8
        if np.any(vals < eps):
            vals_clipped = np.clip(vals, eps, None)
            S = (vecs * vals_clipped).dot(vecs.T)
            S = (S + S.T) / 2.0
    if not np.isfinite(S).all():
        S = np.eye(len(valid_cols)) * 1e-6
    return mu.reindex(valid_cols).fillna(0.0), S, valid_cols


#----------------------------------------------------------------------------------------------------------------------------

In [ ]:
# ensure equal_w defined
equal_w = [1.0/len(obs_vars)] * len(obs_vars)   # obs_cols must be the list of safe obs names used in df
#base_weights = df['base_weights']
T = len(df)
optimized_weights = []  # will have exactly T elements
sl_weights = []

TRAILING_THRESHOLD = -0.05  # e.g., -5% from trailing peak
ENTRY_STREAK = 5  # e.g., 5 consecutive up days to enter
EXIT_STREAK = 5  # e.g., 5 consecutive up days to exit
trailing_peak = 1.0  # Trailing peak for stop loss
streak_up = 0  # Counter for consecutive up days
streak_down = 0  # Counter for consecutive down days
in_market = False  # Flag: True if entered

for t in range(T):
    # for the first WINDOW days, we use equal weights (or base_weights if you prefer)
    if t < WINDOW:
        #optimized_weights.append(list(equal_w))
        optimized_weights.append(list(df['base_weights'][t]))
        sl_weights.append(list(df['base_weights'][t]))
        continue

    # prepare the historical window
    window_df = df.iloc[t-WINDOW:t]
    # Use the safe covariance/mu routine (assumes you implemented safe_cov_and_mu)
    mu, S, valid_cols = safe_cov_and_mu(window_df, obs_vars)  # returns None, None, [] if insufficient
    if S is None or len(valid_cols) == 0:
        optimized_weights.append(list(df['base_weights'][t]))
        continue

    # If some assets were dropped due to NaNs, optimizer uses valid_cols.
    # After optimization, re-map weights back to the full obs_cols order (zeros for dropped assets)
    try:
        ef = EfficientFrontier(mu, S, weight_bounds=(0, 1))
        raw = ef.max_sharpe()
        cleaned = ef.clean_weights()
        #print(cleaned)
        # cleaned is a dict keyed by valid_cols
        w_full = []
        for col in obs_vars:
            w_full.append(float(cleaned.get(col, 0.0)))  # 0 for dropped/absent assets
        #w_full.append(list(cleaned.values()))
    except Exception as e:
        # Log the exception if you want
        #print(f"Optimization failed at t={t}, date={df.index[t]}: {e}")
        w_full = list(df['base_weights'][t])
        
    #----------------------------------------------------------------------------------------------------------
    # Apply stop loss: Check drawdown
    
    # Compute daily return
    daily_return = np.sum(df.iloc[t][obs_vars] * w_full) if t < len(df) else 0  # Hypothetical daily
    
    # Update streak: Based on SP500 as proxy for "market up" (or use daily_return)
    if daily_return > 0:     # Positive day        #df.iloc[i]['^GSPC'] > 0:
        streak_up += 1
        streak_down = 0
    elif daily_return < 0:   # Negative day
        streak_down += 1
        streak_up = 0
    else:                    # Neutral day
        streak_down = 0
        streak_up = 0
    
    # Entry condition: Enter if streak >= ENTRY_STREAK and not in_market
    if streak_up >= ENTRY_STREAK and not in_market:
        in_market = True
        print(f"Entry triggered at {df.index[t]} after {streak_up} up days")


    # Exit on down streak: If streak_down >= EXIT_STREAK and in_market
    if streak_down >= EXIT_STREAK and in_market:
        in_market = False
        weights = SAFE_ALLOCATION.tolist()
        daily_return = 0
        streak_up = 0
        trailing_peak = cum_port  # Reset peak
        print(f"Down streak exit triggered at {df.index[t]} with {streak_down} down days")

    
    # Apply rules
    if not in_market:
        weights = SAFE_ALLOCATION.tolist()  # Stay out
        daily_return = 0  # No exposure
    else: 
        weights = w_full
    
    # Update cumulative and trailing stop
    cum_port *= (1 + daily_return)
    trailing_peak = max(trailing_peak, cum_port)
    trailing_dd = (cum_port - trailing_peak) / trailing_peak if trailing_peak > 0 else 0
    
    #if trailing_dd <= TRAILING_THRESHOLD and in_market:
    #   in_market = False
    #   weights = SAFE_ALLOCATION.tolist()
    #   daily_return = 0
    #   streak_up = 0  # Reset streak
    #   streak_down = 0
    #   trailing_peak = cum_port  # Reset peak after exit
    #   print(f"Trailing stop loss triggered at {df.index[i]} with drawdown {trailing_dd:.2%} from peak")
    #else: 
    #   weights = w_full
    
    optimized_weights.append(w_full)
    sl_weights.append(weights)
    port_returns.append(daily_return)

    #-------------------------------------------------------------------------------------------------------

# final sanity check
assert len(optimized_weights) == len(df), f"expected {len(df)} weights but got {len(optimized_weights)}"

# assign safely
df['optimized_weights'] = optimized_weights

#-----------------------------------------------------------------------------------------------------------------------

In [ ]:
# Compute portfolio returns
#port_returns = pd.Series(np.sum(df[obs_vars].values * np.array(df['optimized_weights'].tolist()), axis=1), index=df.index)

# ---------- PORTFOLIO RETURNS ----------
W = np.vstack(df['optimized_weights'].values)
Y = df[obs_vars].values
port_rets = (W * Y).sum(axis=1)
df['port_returns'] = port_rets
df['cum_port'] = (1 + df['port_returns']).cumprod()


df['sl_weights'] = sl_weights
W_sl = np.vstack(df['sl_weights'].values)
df['ret_sl'] = (W_sl * Y).sum(axis=1)
df['cum_sl'] = (1 + df['ret_sl']).cumprod()
#--------------------------------------------------------------------------------------------------------


# --- Compute base portfolio returns ---
# base_weights is stored as np.array in each row; convert to T x n_assets matrix
W_base = np.vstack(df['base_weights'].values)   # shape (T, n_assets)
#Y = df[obs_vars].values                         # shape (T, n_assets)

df['ret_base'] = (W_base * Y).sum(axis=1)

# --- Compute equal-weight portfolio returns ---
# --- Equal weights ---
equal_w = np.array([1.0/len(obs_vars)] * len(obs_vars))
df['ret_equal'] = (Y * equal_w).sum(axis=1)

# --- Cumulative returns ---
df['cum_base'] = (1 + df['ret_base']).cumprod()
df['cum_equal'] = (1 + df['ret_equal']).cumprod()


#-----------------------------------------------------------------------------------------------------------

In [ ]:
# Plot cumulative returns
#cum_port = (1 + port_returns).cumprod()
plt.figure(figsize=(12, 6))
plt.plot(df['cum_port'], label='Dynamic Overlay Portfolio')
plt.title('Cumulative Returns of Dynamic Overlay')
plt.legend()
plt.savefig("./output/cum_return_DO.png", dpi = 300, bbox_inches = 'tight')

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['cum_port'], label='Dynamic Overlay Portfolio')
plt.plot(df['cum_base'], label='Base Portfolio')
plt.plot(df['cum_equal'], label='Equal Weight Portfolio')
plt.title('Cumulative return - Dynamic Overlay vs Base vs Equal weight')
plt.legend()
plt.savefig("./output/cum_return_comp.png", dpi = 300, bbox_inches = 'tight')

plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['cum_port'], label='Dynamic Overlay Portfolio')
plt.plot(df['cum_sl'], label='Dynamic Overlay Portfolio + Stop Loss')
plt.plot(df['cum_base'], label='Base Portfolio')
plt.plot(df['cum_equal'], label='Equal Weight Portfolio')
plt.fill_between(df.index, df['cum_sl'], df['cum_port'], color='grey', alpha=0.2)
plt.title('Cumulative return - Dynamic Overlay vs Base vs Equal weight')
plt.legend()
plt.savefig("./output/cum_return_comp_stpl.png", dpi = 300, bbox_inches = 'tight')

plt.show()


In [ ]:
# Save updated DataFrame
df.to_parquet('./data/model_results.parquet')
print("Dynamic modeling completed and saved.")